<a href="https://colab.research.google.com/github/ms-starryvoid/ML_Lab_DataSet/blob/main/Programs/NaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import math
from collections import defaultdict

# ----------------------------
# 1. Load Dataset
# ----------------------------
data = pd.read_csv("/content/drive/MyDrive/MTECH S1 /ComputingLab1/Exp2-NaiveBayes/naiv.csv")
print("First 5 rows of the dataset:")
print(data.head())

# Separate features (X) and target (y)
X = data.iloc[:, :-1].values   # all columns except last
y = data.iloc[:, -1].values    # last column

# ----------------------------
# 2. Train/Test Split (80/20)
# ----------------------------
np.random.seed(42)
indices = np.arange(len(X))
np.random.shuffle(indices)

split = int(0.8 * len(X))
train_idx, test_idx = indices[:split], indices[split:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

# ----------------------------
# 3. Gaussian Naive Bayes (from scratch)
# ----------------------------
class GaussianNB_FromScratch:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.priors = {}

        for c in self.classes:
            X_c = X[y == c]
            self.mean[c] = np.mean(X_c, axis=0)
            self.var[c] = np.var(X_c, axis=0) + 1e-6  # avoid zero division
            self.priors[c] = X_c.shape[0] / X.shape[0]

    def gaussian_pdf(self, class_idx, x):
        mean = self.mean[class_idx]
        var = self.var[class_idx]
        numerator = np.exp(-((x - mean) ** 2) / (2 * var))
        denominator = np.sqrt(2 * np.pi * var)
        return numerator / denominator

    def predict_one(self, x):
        posteriors = {}
        for c in self.classes:
            prior = np.log(self.priors[c])
            conditional = np.sum(np.log(self.gaussian_pdf(c, x)))
            posteriors[c] = prior + conditional
        return max(posteriors, key=posteriors.get)

    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])

# ----------------------------
# 4. Train Model
# ----------------------------
model = GaussianNB_FromScratch()
model.fit(X_train, y_train)

# ----------------------------
# 5. Predictions
# ----------------------------
y_pred = model.predict(X_test)

# ----------------------------
# 6. Accuracy
# ----------------------------
accuracy = np.mean(y_pred == y_test)
print(f"\nAccuracy of Naive Bayes classifier: {accuracy * 100:.2f}%")

# ----------------------------
# 7. Show Predictions
# ----------------------------
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print("\nSample Predictions:")
print(comparison.head())

First 5 rows of the dataset:
   glucose  bloodpressure  diabetes
0       40             85         0
1       40             92         0
2       45             63         1
3       45             80         0
4       40             73         1

Accuracy of Naive Bayes classifier: 94.47%

Sample Predictions:
   Actual  Predicted
0       1          1
1       0          0
2       0          1
3       1          1
4       1          1
